# Validation — CaloDiT-2 / CaloChallenge Dataset 2

Implements all validation observables from the CaloDiT-2 paper.
Showers shape convention: **(K, R, PHI, Z)** = (K, 9, 16, 45).

**Requirements:**
- CaloChallenge repo cloned to `./CaloChallenge/` (`git clone https://github.com/OzAmram/CaloChallenge.git`)
- Dataset 2 HDF5 file accessible (e.g. `../calo-data/dataset_2_2.hdf5`)
- Generated showers HDF5 file (or replace the placeholder with model inference)

## 1. Imports & config

In [ ]:
import sys
import numpy as np
import h5py
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams['figure.dpi'] = 110
plt.rcParams['axes.grid'] = False

sys.path.append('./CaloChallenge/code')
from HighLevelFeatures import HighLevelFeatures as HLF

from utilities import (
    CaloChallenge,
    # --- compute functions ---
    compute_long_total_energy, compute_long_total_hits,
    compute_long_first_moment, compute_long_second_moment, compute_long_event_energy,
    compute_rad_total_energy,  compute_rad_total_hits,
    compute_rad_first_moment,  compute_rad_second_moment,  compute_rad_event_energy,
    compute_azim_total_energy, compute_azim_total_hits,
    compute_azim_first_moment, compute_azim_second_moment, compute_azim_event_energy,
    compute_total_event_energy, compute_total_event_hits,
    compute_cell_energy, compute_cell_log_energy,
    # --- plot functions ---
    plot_long_observables, plot_rad_observables,
    plot_azim_observables, plot_global_observables,
    plot_shower_3d, plot_z_profiles, plot_comparison,
)

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
REF_FILE   = '../calo-data/dataset_2_2.hdf5'          # Geant4 reference
GEN_FILE   = '../calo-data/generated_showers.hdf5'    # model output (see §3)
BINNING    = './CaloChallenge/code/binning_dataset_2.xml'
PARTICLE   = 'electron'

# ── Noise threshold matching CaloDiT-2 preprocessing (15.1 keV → MeV) ────────
NOISE_THR  = 1.515e-02   # MeV

# ── How many events to load (None = all) ──────────────────────────────────────
N_SAMPLES  = None

## 2. Load reference showers (Geant4)

In [ ]:
hlf = HLF(PARTICLE, filename=BINNING)

i_idcs = np.arange(N_SAMPLES) if N_SAMPLES is not None else None
ref_ds = CaloChallenge(REF_FILE, hlf, i_idcs=i_idcs)

ref_showers  = ref_ds.showers    # (K, R=9, PHI=16, Z=45)
ref_energies = ref_ds.energies   # (K,) MeV

print(f'Reference — showers: {ref_showers.shape}, energies: {ref_energies.shape}')
print(f'Energy range: {ref_energies.min():.1f} – {ref_energies.max():.1f} MeV')

## 3. Load (or generate) model showers

**Option A** — load from a saved HDF5 file produced by the model's inference script.  
**Option B** — run model inference inline (uncomment the block below).

In [ ]:
# # ── Option A: load pre-generated showers ──────────────────────────────────────
# with h5py.File(GEN_FILE, 'r') as f:
#     print('Keys in generated file:', list(f.keys()))
#     gen_showers  = f['showers'][:]           # expected (K, R, PHI, Z)
#     gen_energies = f['incident_energies'][:].flatten()

# gen_showers  = gen_showers.astype(np.float32)
# gen_energies = gen_energies.astype(np.float32)
# print(f'Generated — showers: {gen_showers.shape}, energies: {gen_energies.shape}')

In [ ]:
# ── Option B: inline inference (uncomment & adapt) ────────────────────────────
from utilities import load_model_from_checkpoint, generate
cfg = {
    'ckpt_dir': '/path/to/checkpoints',
    'device':   'cuda',
    'voxel_shape': (45, 16, 9),   # model's (D, H, W)
}
model, scheduler = load_model_from_checkpoint(cfg, ModelClass, SchedulerClass)
raw = generate(model, scheduler, cfg, inverse_fn, n_samples=1000, e_inc_gev=50.)
gen_showers  = raw[:, 0].transpose(2, 1, 0)   # → (K, R, PHI, Z)
gen_energies = np.full(len(gen_showers), 50e3) # MeV

# ── Fallback: self-comparison (remove before real use) ────────────────────────
# gen_showers  = ref_showers
# gen_energies = ref_energies

MODEL_LABEL = 'DDIM+T'

## 4. Longitudinal profile observables

| Observable | Description |
|---|---|
| LongTotalEnergy | Mean energy deposited per Z-layer |
| LongTotalHits | Mean hit count per Z-layer |
| LongFirstMoment | Energy-weighted mean depth ⟨z⟩ per event |
| LongSecondMoment | Energy-weighted second moment ⟨z²⟩ per event |
| LongEventEnergy | Per-Z-layer energy distribution across events |

In [ ]:
fig_long, fig_long_ev = plot_long_observables(
    ref_showers, gen_showers,
    threshold=NOISE_THR,
    gen_label=MODEL_LABEL,
)
plt.show()

### LongEventEnergy — per-Z-layer energy distributions

In [ ]:
fig_long_ev
plt.show()

### Individual longitudinal compute functions

In [ ]:
ref_long_te = compute_long_total_energy(ref_showers)   # (Z,)
ref_long_th = compute_long_total_hits(ref_showers, NOISE_THR)  # (Z,)
ref_long_fm = compute_long_first_moment(ref_showers)   # (K,)
ref_long_sm = compute_long_second_moment(ref_showers)  # (K,)
ref_long_ee = compute_long_event_energy(ref_showers)   # (K, Z)

print('LongTotalEnergy  shape:', ref_long_te.shape, '  sum:', ref_long_te.sum())
print('LongTotalHits    shape:', ref_long_th.shape)
print('LongFirstMoment  shape:', ref_long_fm.shape, '  mean:', ref_long_fm.mean())
print('LongSecondMoment shape:', ref_long_sm.shape, '  mean:', ref_long_sm.mean())
print('LongEventEnergy  shape:', ref_long_ee.shape)

## 5. Radial profile observables

| Observable | Description |
|---|---|
| RadTotalEnergy | Mean energy deposited per R-bin |
| RadTotalHits | Mean hit count per R-bin |
| RadFirstMoment | Energy-weighted mean radius ⟨r⟩ per event |
| RadSecondMoment | Energy-weighted second moment ⟨r²⟩ per event |
| RadEventEnergy | Per-R-bin energy distribution (3×3 grid for R=9) |

In [ ]:
fig_rad, fig_rad_ev = plot_rad_observables(
    ref_showers, gen_showers,
    threshold=NOISE_THR,
    gen_label=MODEL_LABEL,
)
plt.show()

### RadEventEnergy — per-R-bin energy distributions

In [ ]:
fig_rad_ev
plt.show()

### Individual radial compute functions

In [ ]:
ref_rad_te = compute_rad_total_energy(ref_showers)   # (R,)
ref_rad_th = compute_rad_total_hits(ref_showers, NOISE_THR)  # (R,)
ref_rad_fm = compute_rad_first_moment(ref_showers)   # (K,)
ref_rad_sm = compute_rad_second_moment(ref_showers)  # (K,)
ref_rad_ee = compute_rad_event_energy(ref_showers)   # (K, R)

print('RadTotalEnergy  shape:', ref_rad_te.shape)
print('RadTotalHits    shape:', ref_rad_th.shape)
print('RadFirstMoment  shape:', ref_rad_fm.shape, '  mean:', ref_rad_fm.mean())
print('RadSecondMoment shape:', ref_rad_sm.shape)
print('RadEventEnergy  shape:', ref_rad_ee.shape)

## 6. Azimuthal profile observables

| Observable | Description |
|---|---|
| AzimTotalEnergy | Mean energy deposited per φ-bin |
| AzimTotalHits | Mean hit count per φ-bin |
| AzimFirstMoment | Energy-weighted mean azimuth ⟨φ⟩ per event |
| AzimSecondMoment | Energy-weighted second moment ⟨φ²⟩ per event |
| AzimEventEnergy | Per-φ-bin energy distribution (4×4 grid for PHI=16) |

In [ ]:
fig_azim, fig_azim_ev = plot_azim_observables(
    ref_showers, gen_showers,
    threshold=NOISE_THR,
    gen_label=MODEL_LABEL,
)
plt.show()

### AzimEventEnergy — per-φ-bin energy distributions

In [ ]:
fig_azim_ev
plt.show()

### Individual azimuthal compute functions

In [ ]:
ref_azim_te = compute_azim_total_energy(ref_showers)   # (PHI,)
ref_azim_th = compute_azim_total_hits(ref_showers, NOISE_THR)  # (PHI,)
ref_azim_fm = compute_azim_first_moment(ref_showers)   # (K,)
ref_azim_sm = compute_azim_second_moment(ref_showers)  # (K,)
ref_azim_ee = compute_azim_event_energy(ref_showers)   # (K, PHI)

print('AzimTotalEnergy  shape:', ref_azim_te.shape)
print('AzimTotalHits    shape:', ref_azim_th.shape)
print('AzimFirstMoment  shape:', ref_azim_fm.shape, '  mean:', ref_azim_fm.mean())
print('AzimSecondMoment shape:', ref_azim_sm.shape)
print('AzimEventEnergy  shape:', ref_azim_ee.shape)

## 7. Global shower observables

| Observable | Description |
|---|---|
| TotalEventEnergy | Total energy deposited per event |
| TotalEventHits | Total hit count per event |
| CellEnergy | Individual cell energy (linear x-axis) |
| CellLogEnergy | log₁₀ of individual cell energies |
| CellEnergy_xlog | Individual cell energy (log x-axis) |

In [ ]:
fig_global = plot_global_observables(
    ref_showers, gen_showers,
    threshold=NOISE_THR,
    gen_label=MODEL_LABEL,
)
plt.show()

### Individual global compute functions

In [ ]:
ref_total_e  = compute_total_event_energy(ref_showers)  # (K,)
ref_total_h  = compute_total_event_hits(ref_showers, NOISE_THR)  # (K,)
ref_cell_e   = compute_cell_energy(ref_showers)          # (M,) non-zero cells
ref_cell_le  = compute_cell_log_energy(ref_showers)      # (M,)

print('TotalEventEnergy shape:', ref_total_e.shape,  '  mean:', ref_total_e.mean())
print('TotalEventHits   shape:', ref_total_h.shape,  '  mean:', ref_total_h.mean())
print('CellEnergy       shape:', ref_cell_e.shape,   '  max: ', ref_cell_e.max())
print('CellLogEnergy    shape:', ref_cell_le.shape)

## 8. Energy-stratified comparison (existing utility)

Energy-distribution + radial-profile comparison at discrete energy levels, with EMD scores.

In [ ]:
fig_cmp = plot_comparison(
    ref_showers, ref_energies,
    gen_showers, gen_energies,
    gen_label=MODEL_LABEL,
    n_cols=3,
)
plt.show()

## 9. Depth-profile comparison

In [ ]:
fig_zp, _ = plot_z_profiles(
    ref_showers, gen_showers=gen_showers,
    filename='Dataset 2',
    transverse_axes=(1, 2),  # sum over R and PHI
)
plt.show()

## 10. 3D shower visualization

In [ ]:
IDX = 0   # event index

print('--- Reference shower ---')
fig_3d_ref = plot_shower_3d(
    ref_showers[IDX],
    i_idx=IDX,
    incident_energy=ref_energies[IDX],
    threshold=NOISE_THR,
)
fig_3d_ref.show()

In [ ]:
print('--- Generated shower ---')
fig_3d_gen = plot_shower_3d(
    gen_showers[IDX],
    i_idx=IDX,
    incident_energy=gen_energies[IDX],
    threshold=NOISE_THR,
)
fig_3d_gen.show()